# Notebook 06 — Multimodal Latent-State Model + Hidden Wellness Index (HWI)

This notebook implements the proposed multimodal latent-state architecture following the classical, 1D-CNN, and CNN-BiLSTM benchmarks.

Main idea:

```text
Raw multimodal physiological windows
        ↓
Masked-sensor self-supervised encoder
        ↓
Latent physiological embedding
        ↓
Hidden Wellness Index (HWI)
        ↓
Stress classification + reconstruction + future-state prediction
```

Outputs saved to Google Drive:

- LOSO per-subject results
- pooled predictions
- pooled confusion matrix
- latent embeddings
- HWI values
- model comparison table
- figures


In [ ]:
# ============================================================
# STEP 1. Imports and reproducibility
# ============================================================

import os
import json
import time
import random
import pickle
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, roc_curve, auc
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

try:
    import umap
    UMAP_AVAILABLE = True
except Exception:
    UMAP_AVAILABLE = False

import tensorflow as tf
from tensorflow.keras import layers, models, callbacks, optimizers

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow version:", tf.__version__)
print("GPU devices:", tf.config.list_physical_devices("GPU"))

In [ ]:
# ============================================================
# STEP 2. Project paths
# ============================================================

try:
    from google.colab import drive
    if not os.path.exists("/content/drive/MyDrive"):
        drive.mount("/content/drive")
except Exception as e:
    print("Google Drive mount skipped or already mounted:", e)

PROJECT_ROOT = "/content/drive/MyDrive/Apple_Hidden_Wellness_AI"
DATA_DIR = os.path.join(PROJECT_ROOT, "data", "processed", "raw_multimodal_windows")
RESULTS_DIR = os.path.join(PROJECT_ROOT, "results")
FIGURES_DIR = os.path.join(PROJECT_ROOT, "figures")
MODEL_DIR = os.path.join(PROJECT_ROOT, "models", "notebook06_latent_state_hwi")

for d in [RESULTS_DIR, FIGURES_DIR, MODEL_DIR]:
    os.makedirs(d, exist_ok=True)

NPZ_PATH = os.path.join(DATA_DIR, "wesad_raw_multimodal_windows_32hz_60s_30s.npz")
METADATA_PATH = os.path.join(DATA_DIR, "wesad_raw_multimodal_windows_metadata.json")

print("NPZ_PATH:", NPZ_PATH)
print("Exists:", os.path.exists(NPZ_PATH))
print("METADATA_PATH:", METADATA_PATH)
print("Exists:", os.path.exists(METADATA_PATH))

In [ ]:
# ============================================================
# STEP 3. Load raw multimodal windows from 04A
# ============================================================

if not os.path.exists(NPZ_PATH):
    raise FileNotFoundError(
        f"Could not find {NPZ_PATH}. Run Notebook 04A first and confirm the saved .npz path."
    )

npz = np.load(NPZ_PATH, allow_pickle=True)
print("Keys in NPZ:", list(npz.keys()))

# Robust key recovery
X = npz["X"] if "X" in npz else npz["X_raw"]
y = npz["y"] if "y" in npz else npz["labels"]
subjects = npz["subjects"]

X = X.astype("float32")
y = y.astype("int64")
subjects = subjects.astype(str)

if os.path.exists(METADATA_PATH):
    with open(METADATA_PATH, "r") as f:
        metadata = json.load(f)
else:
    metadata = {}

CHANNELS = metadata.get("channels", ["ACC_x", "ACC_y", "ACC_z", "ECG", "EDA", "EMG", "RESP", "TEMP"])
SAMPLING_RATE = int(metadata.get("sampling_rate_hz", 32))
WINDOW_SECONDS = int(metadata.get("window_seconds", 60))
STEP_SECONDS = int(metadata.get("step_seconds", 30))

N_WINDOWS, WINDOW_SAMPLES, N_CHANNELS = X.shape
unique_subjects = np.array(sorted(np.unique(subjects), key=lambda s: int(s.replace("S", ""))))

print("X shape:", X.shape)
print("y shape:", y.shape)
print("subjects shape:", subjects.shape)
print("Subjects:", unique_subjects)
print("Class counts:", dict(zip(*np.unique(y, return_counts=True))))
print("Channels:", CHANNELS)

In [ ]:
# ============================================================
# STEP 4. Create future-window targets within each subject
# ============================================================
# For every window, the future target is the next window from the same subject.
# The last window of each subject is excluded because it has no future target.

valid_indices = []
future_indices = []

for s in unique_subjects:
    idx = np.where(subjects == s)[0]
    idx = np.sort(idx)
    if len(idx) < 2:
        continue
    valid_indices.extend(idx[:-1].tolist())
    future_indices.extend(idx[1:].tolist())

valid_indices = np.array(valid_indices, dtype=int)
future_indices = np.array(future_indices, dtype=int)

X_current = X[valid_indices]
X_future = X[future_indices]
y_current = y[valid_indices]
subjects_current = subjects[valid_indices]

print("Current X:", X_current.shape)
print("Future X:", X_future.shape)
print("Labels:", y_current.shape)
print("Subjects:", subjects_current.shape)
print("Class counts after future alignment:", dict(zip(*np.unique(y_current, return_counts=True))))

In [ ]:
# ============================================================
# STEP 5. Utility functions
# ============================================================

def standardize_by_training_subjects(X_train, X_val, X_test):
    """Channel-wise standardization using training fold only."""
    mean = X_train.reshape(-1, X_train.shape[-1]).mean(axis=0)
    std = X_train.reshape(-1, X_train.shape[-1]).std(axis=0)
    std[std == 0] = 1.0
    return (X_train - mean) / std, (X_val - mean) / std, (X_test - mean) / std, mean, std


def apply_standardization(X_arr, mean, std):
    return (X_arr - mean) / std


def make_masked_input(X_arr, mask_probability=0.25, seed=None):
    """Mask complete sensor channels per sample, preserving the target as original X."""
    rng = np.random.default_rng(seed)
    X_masked = X_arr.copy()
    n_samples, _, n_channels = X_arr.shape
    mask_matrix = rng.random((n_samples, n_channels)) < mask_probability

    # Make sure at least one channel remains unmasked for each sample.
    for i in range(n_samples):
        if mask_matrix[i].all():
            keep_ch = rng.integers(0, n_channels)
            mask_matrix[i, keep_ch] = False

    for i in range(n_samples):
        X_masked[i, :, mask_matrix[i]] = 0.0

    return X_masked.astype("float32"), mask_matrix.astype("int64")


def choose_validation_subjects(train_subjects, n_val_subjects=2, seed=42):
    rng = np.random.default_rng(seed)
    train_subjects = np.array(sorted(train_subjects, key=lambda s: int(s.replace("S", ""))))
    n_val_subjects = min(n_val_subjects, max(1, len(train_subjects) // 5))
    return rng.choice(train_subjects, size=n_val_subjects, replace=False)


def safe_auc(y_true, y_prob):
    try:
        if len(np.unique(y_true)) < 2:
            return np.nan
        return roc_auc_score(y_true, y_prob)
    except Exception:
        return np.nan


def compute_metrics(y_true, y_prob, threshold=0.5):
    y_pred = (y_prob >= threshold).astype(int)
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "roc_auc": safe_auc(y_true, y_prob),
    }


def savefig(name):
    path_png = os.path.join(FIGURES_DIR, name + ".png")
    path_pdf = os.path.join(FIGURES_DIR, name + ".pdf")
    plt.savefig(path_png, dpi=300, bbox_inches="tight")
    plt.savefig(path_pdf, bbox_inches="tight")
    print("Saved figure:", path_png)

In [ ]:
# ============================================================
# STEP 6. Build latent-state HWI model
# ============================================================

LATENT_DIM = 64
LEARNING_RATE = 1e-3
MASK_PROBABILITY = 0.25

# Relative weights for stress classification, masked reconstruction,
# and next-window physiological prediction.
LOSS_WEIGHTS = [1.0, 0.20, 0.20]


def conv_block(x, filters, kernel_size, pool_size=2, dropout=0.15):
    x = layers.Conv1D(filters, kernel_size, padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Conv1D(filters, kernel_size, padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling1D(pool_size=pool_size)(x)
    x = layers.Dropout(dropout)(x)
    return x


def build_latent_hwi_model(input_shape, latent_dim=64, learning_rate=1e-3):
    inp = layers.Input(shape=input_shape, name="physiological_window_input")

    x = conv_block(inp, 32, 9, pool_size=2, dropout=0.10)
    x = conv_block(x, 64, 7, pool_size=2, dropout=0.15)
    x = conv_block(x, 96, 5, pool_size=2, dropout=0.20)
    x = layers.Conv1D(128, 3, padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.25)(x)

    z = layers.Dense(latent_dim, activation="linear", name="latent_embedding")(x)
    z_norm = layers.LayerNormalization(name="latent_embedding_norm")(z)

    # Scalar latent projection used as the preliminary Hidden Wellness Index.
# The sigmoid activation constrains the output to the interval [0, 1].
    hwi = layers.Dense(1, activation="sigmoid", name="hidden_wellness_index")(z_norm)

   # Supervised classification head for baseline-versus-stress prediction.
cls = layers.Dense(32, activation="relu")(z_norm)
    cls = layers.Dropout(0.20)(cls)
    stress_out = layers.Dense(1, activation="sigmoid", name="stress_output")(cls)

    # Decoder for reconstructing masked channels in the current physiological window.
    rec = layers.Dense(256, activation="relu")(z_norm)
    rec = layers.Dropout(0.20)(rec)
    rec_size = int(np.prod(input_shape))
    rec = layers.Dense(rec_size, activation="linear")(rec)
    rec_out = layers.Reshape(input_shape, name="reconstruction_output")(rec)

    # Decoder for predicting the subsequent physiological window from the current latent state.
    fut = layers.Dense(256, activation="relu")(z_norm)
    fut = layers.Dropout(0.20)(fut)
    fut_size = int(np.prod(input_shape))
    fut = layers.Dense(fut_size, activation="linear")(fut)
    future_out = layers.Reshape(input_shape, name="future_output")(fut)

    model = models.Model(
        inputs=inp,
        outputs=[stress_out, rec_out, future_out],
        name="multimodal_latent_state_hwi_model"
    )

    encoder = models.Model(
        inputs=inp,
        outputs=[z_norm, hwi, stress_out],
        name="latent_state_encoder_hwi"
    )

    model.compile(
        optimizer=optimizers.Adam(learning_rate=learning_rate),
        loss=["binary_crossentropy", "mse", "mse"],
        loss_weights=LOSS_WEIGHTS,
        metrics=[["accuracy", tf.keras.metrics.AUC(name="auc")], [], []]
    )

    return model, encoder

preview_model, preview_encoder = build_latent_hwi_model((WINDOW_SAMPLES, N_CHANNELS), LATENT_DIM, LEARNING_RATE)
preview_model.summary()
print("Model output names:", preview_model.output_names)

In [ ]:
# ============================================================
# STEP 7. LOSO training for latent-state HWI model
# ============================================================

MAX_EPOCHS = 35
BATCH_SIZE = 32
THRESHOLD = 0.5
N_VAL_SUBJECTS = 2

# QUICK_RUN limits the analysis to three LOSO folds for implementation checks.
# The full subject-independent evaluation uses QUICK_RUN=False.
QUICK_RUN = False

subjects_to_run = unique_subjects[:3] if QUICK_RUN else unique_subjects

fold_rows = []
all_pred_rows = []
all_latent_rows = []
all_hist_rows = []

start_all = time.time()

for fold_idx, test_subject in enumerate(subjects_to_run, start=1):
    print("\n" + "=" * 80)
    print(f"LOSO fold {fold_idx}/{len(subjects_to_run)} — held-out subject: {test_subject}")
    print("=" * 80)

    train_candidate_subjects = np.array([s for s in unique_subjects if s != test_subject])
    val_subjects = choose_validation_subjects(
        train_candidate_subjects,
        n_val_subjects=N_VAL_SUBJECTS,
        seed=SEED + fold_idx
    )
    train_subjects = np.array([s for s in train_candidate_subjects if s not in val_subjects])

    train_mask = np.isin(subjects_current, train_subjects)
    val_mask = np.isin(subjects_current, val_subjects)
    test_mask = subjects_current == test_subject

    X_train = X_current[train_mask]
    y_train = y_current[train_mask]
    F_train = X_future[train_mask]

    X_val = X_current[val_mask]
    y_val = y_current[val_mask]
    F_val = X_future[val_mask]

    X_test = X_current[test_mask]
    y_test = y_current[test_mask]
    F_test = X_future[test_mask]

    X_train, X_val, X_test, fold_mean, fold_std = standardize_by_training_subjects(X_train, X_val, X_test)
    F_train = apply_standardization(F_train, fold_mean, fold_std)
    F_val = apply_standardization(F_val, fold_mean, fold_std)
    F_test = apply_standardization(F_test, fold_mean, fold_std)

    X_train_masked, train_channel_mask = make_masked_input(X_train, MASK_PROBABILITY, seed=SEED + fold_idx)
    X_val_masked, val_channel_mask = make_masked_input(X_val, MASK_PROBABILITY, seed=SEED + 100 + fold_idx)
    X_test_masked, test_channel_mask = make_masked_input(X_test, MASK_PROBABILITY, seed=SEED + 200 + fold_idx)

    print("Train subjects:", train_subjects)
    print("Val subjects:", val_subjects)
    print("Train/val/test shapes:", X_train.shape, X_val.shape, X_test.shape)
    print("Train class counts:", dict(zip(*np.unique(y_train, return_counts=True))))

    tf.keras.backend.clear_session()
    model, encoder = build_latent_hwi_model((WINDOW_SAMPLES, N_CHANNELS), LATENT_DIM, LEARNING_RATE)

    cb = [
        callbacks.EarlyStopping(monitor="val_loss", patience=7, restore_best_weights=True),
        callbacks.ReduceLROnPlateau(monitor="val_loss", patience=3, factor=0.5, min_lr=1e-5),
    ]

    t0 = time.time()
    hist = model.fit(
        X_train_masked,
        [y_train, X_train, F_train],
        validation_data=(X_val_masked, [y_val, X_val, F_val]),
        epochs=MAX_EPOCHS,
        batch_size=BATCH_SIZE,
        callbacks=cb,
        verbose=1
    )
    train_time = time.time() - t0

    # Predict on masked held-out subject.
    pred_outputs = model.predict(X_test_masked, batch_size=BATCH_SIZE, verbose=0)
    y_prob = pred_outputs[0].reshape(-1)
    rec_pred = pred_outputs[1]
    future_pred = pred_outputs[2]

    z_test, hwi_test, stress_from_encoder = encoder.predict(X_test_masked, batch_size=BATCH_SIZE, verbose=0)
    hwi_test = hwi_test.reshape(-1)

    metrics = compute_metrics(y_test, y_prob, THRESHOLD)
    rec_mse = float(np.mean((rec_pred - X_test) ** 2))
    future_mse = float(np.mean((future_pred - F_test) ** 2))

    row = {
        "fold": fold_idx,
        "test_subject": str(test_subject),
        "n_train": int(len(y_train)),
        "n_val": int(len(y_val)),
        "n_test": int(len(y_test)),
        "train_subjects": ",".join(map(str, train_subjects)),
        "val_subjects": ",".join(map(str, val_subjects)),
        "accuracy": metrics["accuracy"],
        "precision": metrics["precision"],
        "recall": metrics["recall"],
        "f1": metrics["f1"],
        "roc_auc": metrics["roc_auc"],
        "reconstruction_mse": rec_mse,
        "future_mse": future_mse,
        "train_time_seconds": train_time,
        "epochs_ran": len(hist.history.get("loss", [])),
    }
    fold_rows.append(row)
    print("Fold metrics:", row)

    for i in range(len(y_test)):
        all_pred_rows.append({
            "fold": fold_idx,
            "subject": str(test_subject),
            "sample_index_within_fold": int(i),
            "y_true": int(y_test[i]),
            "y_prob_stress": float(y_prob[i]),
            "y_pred": int(y_prob[i] >= THRESHOLD),
            "hwi": float(hwi_test[i]),
            "reconstruction_mse_sample": float(np.mean((rec_pred[i] - X_test[i]) ** 2)),
            "future_mse_sample": float(np.mean((future_pred[i] - F_test[i]) ** 2)),
        })

        latent_row = {
            "fold": fold_idx,
            "subject": str(test_subject),
            "sample_index_within_fold": int(i),
            "y_true": int(y_test[i]),
            "y_prob_stress": float(y_prob[i]),
            "hwi": float(hwi_test[i]),
        }
        for j in range(z_test.shape[1]):
            latent_row[f"z_{j:03d}"] = float(z_test[i, j])
        all_latent_rows.append(latent_row)

    for epoch_idx in range(len(hist.history.get("loss", []))):
        hist_row = {
            "fold": fold_idx,
            "test_subject": str(test_subject),
            "epoch": int(epoch_idx + 1),
        }
        for k, v in hist.history.items():
            hist_row[k] = float(v[epoch_idx])
        all_hist_rows.append(hist_row)

print("\nTotal LOSO time (minutes):", round((time.time() - start_all) / 60, 2))

fold_results_df = pd.DataFrame(fold_rows)
pooled_predictions_df = pd.DataFrame(all_pred_rows)
latent_embeddings_df = pd.DataFrame(all_latent_rows)
training_history_df = pd.DataFrame(all_hist_rows)

fold_results_df

In [ ]:
# ============================================================
# STEP 8. Pooled evaluation
# ============================================================

y_true_all = pooled_predictions_df["y_true"].values.astype(int)
y_prob_all = pooled_predictions_df["y_prob_stress"].values.astype(float)
y_pred_all = pooled_predictions_df["y_pred"].values.astype(int)
hwi_all = pooled_predictions_df["hwi"].values.astype(float)

pooled_metrics = compute_metrics(y_true_all, y_prob_all, THRESHOLD)
cm = confusion_matrix(y_true_all, y_pred_all)

print("Pooled metrics:")
for k, v in pooled_metrics.items():
    print(f"  {k}: {v:.4f}")
print("Confusion matrix:\n", cm)

pooled_summary_df = pd.DataFrame([pooled_metrics])
pooled_summary_df["model"] = "Latent-State HWI"
pooled_summary_df["n_windows"] = len(y_true_all)
pooled_summary_df["threshold"] = THRESHOLD
pooled_summary_df = pooled_summary_df[["model", "n_windows", "threshold", "accuracy", "precision", "recall", "f1", "roc_auc"]]
pooled_summary_df

In [ ]:
# ============================================================
# STEP 9. Plot ROC curve and confusion matrix
# ============================================================

fpr, tpr, _ = roc_curve(y_true_all, y_prob_all)
roc_auc_value = auc(fpr, tpr)

plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, label=f"Latent-State HWI AUC = {roc_auc_value:.3f}")
plt.plot([0, 1], [0, 1], linestyle="--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Notebook 06 LOSO Pooled ROC Curve")
plt.legend(loc="lower right")
plt.grid(alpha=0.3)
savefig("notebook06_latent_hwi_loso_pooled_roc_curve")
plt.show()

plt.figure(figsize=(5, 4))
plt.imshow(cm, interpolation="nearest")
plt.title("Notebook 06 LOSO Pooled Confusion Matrix")
plt.xticks([0, 1], ["Baseline", "Stress"])
plt.yticks([0, 1], ["Baseline", "Stress"])
plt.xlabel("Predicted label")
plt.ylabel("True label")
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(j, i, str(cm[i, j]), ha="center", va="center")
plt.colorbar()
savefig("notebook06_latent_hwi_loso_pooled_confusion_matrix")
plt.show()

In [ ]:
# ============================================================
# STEP 10. HWI analysis
# ============================================================

hwi_df = pooled_predictions_df.copy()

hwi_summary = hwi_df.groupby("y_true")["hwi"].agg(["count", "mean", "std", "min", "median", "max"]).reset_index()
hwi_summary["class_name"] = hwi_summary["y_true"].map({0: "Baseline", 1: "Stress"})
hwi_summary

In [ ]:
plt.figure(figsize=(6, 4))
plt.boxplot(
    [hwi_df.loc[hwi_df["y_true"] == 0, "hwi"], hwi_df.loc[hwi_df["y_true"] == 1, "hwi"]],
    labels=["Baseline", "Stress"],
    showfliers=False
)
plt.ylabel("Hidden Wellness Index (HWI)")
plt.title("HWI Distribution by Class")
plt.grid(axis="y", alpha=0.3)
savefig("notebook06_hwi_distribution_by_class")
plt.show()

In [ ]:
# ============================================================
# STEP 11. Latent-space visualization using PCA/UMAP
# ============================================================

z_cols = [c for c in latent_embeddings_df.columns if c.startswith("z_")]
Z = latent_embeddings_df[z_cols].values
labels_latent = latent_embeddings_df["y_true"].values

pca = PCA(n_components=2, random_state=SEED)
Z_pca = pca.fit_transform(Z)
latent_embeddings_df["pca1"] = Z_pca[:, 0]
latent_embeddings_df["pca2"] = Z_pca[:, 1]

plt.figure(figsize=(6, 5))
for cls, name in [(0, "Baseline"), (1, "Stress")]:
    mask = labels_latent == cls
    plt.scatter(Z_pca[mask, 0], Z_pca[mask, 1], s=18, alpha=0.7, label=name)
plt.xlabel("Latent PC1")
plt.ylabel("Latent PC2")
plt.title("Latent Physiological Space — PCA")
plt.legend()
plt.grid(alpha=0.3)
savefig("notebook06_latent_space_pca")
plt.show()

if UMAP_AVAILABLE:
    reducer = umap.UMAP(n_neighbors=20, min_dist=0.1, random_state=SEED)
    Z_umap = reducer.fit_transform(Z)
    latent_embeddings_df["umap1"] = Z_umap[:, 0]
    latent_embeddings_df["umap2"] = Z_umap[:, 1]

    plt.figure(figsize=(6, 5))
    for cls, name in [(0, "Baseline"), (1, "Stress")]:
        mask = labels_latent == cls
        plt.scatter(Z_umap[mask, 0], Z_umap[mask, 1], s=18, alpha=0.7, label=name)
    plt.xlabel("UMAP 1")
    plt.ylabel("UMAP 2")
    plt.title("Latent Physiological Space — UMAP")
    plt.legend()
    plt.grid(alpha=0.3)
    savefig("notebook06_latent_space_umap")
    plt.show()
else:
    print("UMAP is not installed. PCA figure was generated instead.")

In [ ]:
# ============================================================
# STEP 12. Compare Notebook 06 with previous baselines if available
# ============================================================

comparison_rows = []

# Add Notebook 06 result
row06 = pooled_summary_df.iloc[0].to_dict()
comparison_rows.append(row06)

# Try to load Notebook 05 and 05B summaries if present.
summary_candidates = [
    ("1D CNN", os.path.join(RESULTS_DIR, "wesad_1dcnn_loso_pooled_summary.csv")),
    ("CNN-BiLSTM", os.path.join(RESULTS_DIR, "wesad_cnn_bilstm05B_loso_pooled_summary.csv")),
    ("Random Forest", os.path.join(RESULTS_DIR, "wesad_loso_summary_results.csv")),
]

for model_name, path in summary_candidates:
    if os.path.exists(path):
        try:
            df_tmp = pd.read_csv(path)
            tmp = df_tmp.iloc[0].to_dict()
            tmp["model"] = model_name
            comparison_rows.append(tmp)
            print("Loaded:", path)
        except Exception as e:
            print("Could not load", path, e)

model_comparison_df = pd.DataFrame(comparison_rows)

# Keep common columns when available.
preferred_cols = ["model", "accuracy", "precision", "recall", "f1", "roc_auc", "n_windows", "threshold"]
existing_cols = [c for c in preferred_cols if c in model_comparison_df.columns]
model_comparison_df = model_comparison_df[existing_cols + [c for c in model_comparison_df.columns if c not in existing_cols]]
model_comparison_df

In [ ]:
# ============================================================
# STEP 13. Save all Notebook 06 outputs
# ============================================================

prefix = "wesad_latent_hwi06"

fold_results_path = os.path.join(RESULTS_DIR, f"{prefix}_loso_by_subject_results.csv")
pooled_predictions_path = os.path.join(RESULTS_DIR, f"{prefix}_loso_pooled_predictions.csv")
pooled_summary_path = os.path.join(RESULTS_DIR, f"{prefix}_loso_pooled_summary.csv")
confusion_matrix_path = os.path.join(RESULTS_DIR, f"{prefix}_loso_confusion_matrix.csv")
latent_embeddings_path = os.path.join(RESULTS_DIR, f"{prefix}_latent_embeddings_hwi.csv")
hwi_summary_path = os.path.join(RESULTS_DIR, f"{prefix}_hwi_summary.csv")
training_history_path = os.path.join(RESULTS_DIR, f"{prefix}_loso_training_history.csv")
model_comparison_path = os.path.join(RESULTS_DIR, f"{prefix}_model_comparison.csv")
config_path = os.path.join(RESULTS_DIR, f"{prefix}_config_and_summary.json")
manifest_path = os.path.join(RESULTS_DIR, f"wesad_06_latent_hwi_output_manifest.csv")

fold_results_df.to_csv(fold_results_path, index=False)
pooled_predictions_df.to_csv(pooled_predictions_path, index=False)
pooled_summary_df.to_csv(pooled_summary_path, index=False)
pd.DataFrame(cm, index=["true_baseline", "true_stress"], columns=["pred_baseline", "pred_stress"]).to_csv(confusion_matrix_path)
latent_embeddings_df.to_csv(latent_embeddings_path, index=False)
hwi_summary.to_csv(hwi_summary_path, index=False)
training_history_df.to_csv(training_history_path, index=False)
model_comparison_df.to_csv(model_comparison_path, index=False)

config = {
    "notebook": "06_wesad_multimodal_latent_state_hidden_wellness_index",
    "raw_window_npz": NPZ_PATH,
    "metadata_json": METADATA_PATH,
    "n_original_windows": int(N_WINDOWS),
    "n_aligned_windows_with_future_target": int(len(y_current)),
    "window_samples": int(WINDOW_SAMPLES),
    "n_channels": int(N_CHANNELS),
    "channels": CHANNELS,
    "subjects": [str(s) for s in unique_subjects],
    "class_counts_aligned": {str(k): int(v) for k, v in zip(*np.unique(y_current, return_counts=True))},
    "evaluation": "Leave-One-Subject-Out",
    "validation_strategy": "two training subjects held out for validation in each fold",
    "architecture": "CNN multimodal encoder with latent embedding, HWI branch, stress head, masked reconstruction head, and future-window prediction head",
    "latent_dim": int(LATENT_DIM),
    "mask_probability": float(MASK_PROBABILITY),
    "loss_weights": LOSS_WEIGHTS,
    "max_epochs": int(MAX_EPOCHS),
    "batch_size": int(BATCH_SIZE),
    "learning_rate": float(LEARNING_RATE),
    "threshold": float(THRESHOLD),
    "quick_run": bool(QUICK_RUN),
    "pooled_metrics": {k: float(v) for k, v in pooled_metrics.items()},
    "mean_reconstruction_mse": float(pooled_predictions_df["reconstruction_mse_sample"].mean()),
    "mean_future_mse": float(pooled_predictions_df["future_mse_sample"].mean()),
    "hwi_summary": hwi_summary.to_dict(orient="records"),
}

with open(config_path, "w") as f:
    json.dump(config, f, indent=2)

saved_files = [
    fold_results_path,
    pooled_predictions_path,
    pooled_summary_path,
    confusion_matrix_path,
    latent_embeddings_path,
    hwi_summary_path,
    training_history_path,
    model_comparison_path,
    config_path,
]
manifest_df = pd.DataFrame({"saved_file": saved_files})
manifest_df.to_csv(manifest_path, index=False)

print("Saved Notebook 06 outputs to:", RESULTS_DIR)
print(manifest_df)

## Interpretation and next analyses

The model produces four complementary outputs:

1. Latent physiological embeddings.
2. A scalar latent-derived Hidden Wellness Index.
3. Masked-channel reconstructions.
4. Next-window physiological predictions.

Accordingly, evaluation should not rely exclusively on stress-classification performance. The analysis also considers reconstruction error, future-state prediction error, cross-subject generalization, latent-space structure, and HWI behavior across experimental conditions.

The next methodological step is an ablation analysis comparing:

- classification only,
- classification with reconstruction,
- classification with future-state prediction,
- the complete multi-objective model.

This comparison will quantify the contribution of each auxiliary objective to classification, representation quality, and cross-subject generalization.